In [1]:
from pathlib import Path
import pandas as pd

D = Path("/home/seowon1211/data1")

paths = {
    "in vitro": D / "vitro_train_test.csv",
    "in vivo": D / "vivo_train_test.csv",
}

metadata_candidates = [
    "BARCODE",
    "COMPOUND_NAME",
    "EXP_TEST_TYPE",
    "SINGLE_REPEAT_TYPE",
    "SACRIFICE_PERIOD",
    "DOSE",
    "DOSE_UNIT",
    "DOSE_LEVEL",
    "usage",
]

for name, path in paths.items():
    assert path.exists(), f"파일 없음: {path}"

    header = pd.read_csv(path, nrows=0).columns.tolist()

    metadata = [
        c for c in metadata_candidates
        if c in header
    ]
    probes = [
        c for c in header
        if str(c).endswith("_at")
    ]

    # 구조 확인용으로 일부 probe와 앞쪽 행만 읽음
    preview = pd.read_csv(
        path,
        usecols=metadata + probes[:5],
        nrows=200,
        low_memory=False,
    )

    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print("전체 컬럼 수:", len(header))
    print("probe 컬럼 수:", len(probes))
    print("메타데이터 컬럼:", metadata)

    print("\n처리 조건 예시:")
    condition_cols = [
        c for c in [
            "COMPOUND_NAME",
            "SACRIFICE_PERIOD",
            "DOSE",
            "DOSE_UNIT",
            "DOSE_LEVEL",
        ]
        if c in preview
    ]

    display(
        preview[condition_cols]
        .drop_duplicates()
        .head(20)
    )

    print("\nprobe 발현값 예시:")
    display(preview[probes[:5]].head())

    print("\n앞쪽 200행에서의 값 형식:")
    print(preview[metadata].dtypes)

print(
    "\n구조 파일 존재:",
    (D / "struct_feats/struct_morgan2048.csv").exists(),
)

In [2]:
from pathlib import Path
import pandas as pd

D = Path("/home/seowon1211/data1")
genes_path = D / "AIVIVE/data/final_rat_genes.csv"

genes = pd.read_csv(genes_path)
assert "PROBEID" in genes.columns

requested = set(genes.PROBEID.dropna().astype(str))

for name in ["vitro", "vivo"]:
    path = D / f"{name}_train_test.csv"
    header = pd.read_csv(path, nrows=0).columns
    probes = [c for c in header if c in requested]

    columns = [
        "COMPOUND_NAME",
        "SACRIFICE_PERIOD",
        "DOSE",
        "DOSE_UNIT",
        "DOSE_LEVEL",
        "usage",
        "EXP_TEST_TYPE",
        "SINGLE_REPEAT_TYPE",
    ]

    meta = pd.read_csv(
        path,
        usecols=[c for c in columns if c in header],
        low_memory=False,
    )

    print(f"\n=== {name} 전체 조건 ===")
    print("샘플 수:", len(meta))
    print("화합물 수:", meta.COMPOUND_NAME.nunique())
    print("사용 가능한 AIVIVE probe 수:", len(probes))

    for col in [
        "SACRIFICE_PERIOD",
        "DOSE_UNIT",
        "usage",
        "EXP_TEST_TYPE",
        "SINGLE_REPEAT_TYPE",
    ]:
        if col in meta:
            print(f"\n{col}:")
            print(meta[col].value_counts(dropna=False).to_string())

    dose = pd.to_numeric(meta.DOSE, errors="coerce")
    print("\n숫자로 해석할 수 없는 용량 행:", int(dose.isna().sum()))

    print("\n단위별 용량 범위:")
    display(
        meta.assign(numeric_dose=dose)
        .groupby("DOSE_UNIT", dropna=False)
        .agg(
            rows=("COMPOUND_NAME", "size"),
            min_dose=("numeric_dose", "min"),
            max_dose=("numeric_dose", "max"),
        )
    )

In [3]:
from pathlib import Path
import pandas as pd

D = Path("/home/seowon1211/data1")

for name in ["vitro", "vivo"]:
    meta = pd.read_csv(
        D / f"{name}_train_test.csv",
        usecols=[
            "COMPOUND_NAME",
            "DOSE",
            "DOSE_UNIT",
            "DOSE_LEVEL",
        ],
        low_memory=False,
    )

    conditions = (
        meta.drop_duplicates()
        .sort_values(["DOSE_UNIT", "COMPOUND_NAME", "DOSE"])
    )

    print(f"\n=== {name}: 단위의 정확한 문자 ===")
    for unit in meta.DOSE_UNIT.dropna().unique():
        print(
            repr(unit),
            "문자 코드:",
            " ".join(f"U+{ord(c):04X}" for c in unit),
        )

    print("\n단위별 대표 조건:")
    display(
        conditions.groupby("DOSE_UNIT", dropna=False)
        .head(8)
    )

    conditions.to_csv(
        D / f"{name}_dose_unit_audit.csv",
        index=False,
    )

In [4]:
import subprocess

result = subprocess.run(
    [
        "rg", "-n", "-i",
        r"DOSE_UNIT|encoding|latin|cp125|shift.?jis|µ|μ|ug/ml|umol",
        "/home/seowon1211/data1/AIVIVE/preprocessing",
        "/home/seowon1211/data1/AIVIVE/README.md",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout or "관련 문자열을 찾지 못했습니다.")
print(result.stderr)

In [5]:
from pathlib import Path
import shutil
import subprocess
import pandas as pd

D = Path("/home/seowon1211/data1")

for name in ["vitro", "vivo"]:
    meta = pd.read_csv(
        D / f"{name}_train_test.csv",
        usecols=[
            "BARCODE",
            "COMPOUND_NAME",
            "SACRIFICE_PERIOD",
            "DOSE",
            "DOSE_UNIT",
            "DOSE_LEVEL",
        ],
        # CSV에 저장된 숫자 표기를 그대로 확인
        dtype=str,
    )

    print(f"\n=== {name}: LPS 전체 조건 ===")
    display(
        meta.loc[meta.COMPOUND_NAME.eq("LPS")]
        .drop_duplicates()
    )

    dose = pd.to_numeric(meta.DOSE, errors="coerce")
    treated_zero = meta[
        dose.eq(0)
        & ~meta.DOSE_LEVEL.str.lower().eq("control")
    ]

    print("\n처리군인데 용량이 0인 조건:")
    display(
        treated_zero.drop(columns="BARCODE")
        .drop_duplicates()
    )

# 앞선 검색이 실제로 실행됐는지도 확인
rg = shutil.which("rg")

if rg:
    result = subprocess.run(
        [
            rg, "-n", "-i",
            r"dose|astype|round|encoding|read_csv",
            str(D / "AIVIVE/preprocessing"),
        ],
        capture_output=True,
        text=True,
    )
    print("\n=== 전처리 코드 검색 ===")
    print("검색 종료 코드:", result.returncode)
    print(result.stdout or "일치하는 내용 없음")
    print(result.stderr)
else:
    print("\nrg가 설치되어 있지 않아 코드 검색을 실행하지 못했습니다.")

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

D = Path("/home/seowon1211/data1")
SOURCE = D / "tggates_metadata/Open-tggates_AllAttribute.tsv"
OUT = D / "conditioned_expression/metadata"
OUT.mkdir(parents=True, exist_ok=True)

assert SOURCE.exists(), f"파일 없음: {SOURCE}"

# 확인한 원본 파일의 인코딩
reference = pd.read_csv(
    SOURCE,
    sep="\t",
    encoding="cp932",
    dtype=str,
    keep_default_na=False,
)

required = {
    "BARCODE", "COMPOUND_NAME", "DOSE",
    "DOSE_UNIT", "DOSE_LEVEL", "SACRI_PERIOD",
}
assert required.issubset(reference.columns)


def barcode_key(series):
    # 숫자 식별자만 허용하고 앞자리 0 제거
    s = series.astype(str).str.strip()
    return s.str.lstrip("0").where(
        s.str.fullmatch(r"\d+") & s.str.lstrip("0").ne("")
    )


reference["barcode_key"] = barcode_key(reference.BARCODE)

# No ChipData 등 실제 칩 식별자가 없는 행 제외
reference = reference[
    reference.barcode_key.notna()
].copy()

assert not reference.barcode_key.duplicated().any(), (
    "원본 메타데이터에 BARCODE 중복이 있습니다."
)

reference = reference[
    [
        "barcode_key",
        "COMPOUND_NAME",
        "DOSE",
        "DOSE_UNIT",
        "DOSE_LEVEL",
        "SACRI_PERIOD",
    ]
].rename(columns={
    "COMPOUND_NAME": "compound_reference",
    "DOSE": "dose_reference",
    "DOSE_UNIT": "dose_unit_reference",
    "DOSE_LEVEL": "dose_level_reference",
    "SACRI_PERIOD": "time_reference",
})

for name in ["vitro", "vivo"]:
    path = D / f"{name}_train_test.csv"

    # 大容量の発現値は読み込まず、メタデータだけ取得
    meta = pd.read_csv(
        path,
        usecols=[
            "BARCODE",
            "COMPOUND_NAME",
            "SACRIFICE_PERIOD",
            "DOSE",
            "DOSE_UNIT",
            "DOSE_LEVEL",
            "usage",
        ],
        dtype=str,
        keep_default_na=False,
    )

    meta.insert(0, "expression_row", np.arange(len(meta)))
    meta["barcode_key"] = barcode_key(meta.BARCODE)

    assert meta.barcode_key.notna().all(), (
        f"{name}: 不正なBARCODEがあります。"
    )

    result = meta.merge(
        reference,
        on="barcode_key",
        how="left",
        validate="many_to_one",
        indicator=True,
        sort=False,
    )

    unmatched = result[result["_merge"].ne("both")]
    unmatched.to_csv(
        OUT / f"{name}_unmatched.csv", index=False
    )

    assert unmatched.empty, (
        f"{name}: 未接続が{len(unmatched)}行あります。"
    )
    assert np.array_equal(
        result.expression_row.to_numpy(),
        np.arange(len(meta)),
    ), "元データの行順が変わりました。"

    # 時間と用量等級が同じか確認
    for original, corrected in [
        ("SACRIFICE_PERIOD", "time_reference"),
        ("DOSE_LEVEL", "dose_level_reference"),
    ]:
        mismatch = (
            result[original].str.strip()
            != result[corrected].str.strip()
        )

        if mismatch.any():
            result.loc[mismatch].to_csv(
                OUT / f"{name}_{original}_mismatch.csv",
                index=False,
            )
            raise ValueError(
                f"{name}: {original}が一致しない行があります。"
            )

    result["dose_numeric_corrected"] = pd.to_numeric(
        result.dose_reference,
        errors="raise",
    )

    assert np.isfinite(result.dose_numeric_corrected).all()
    assert result.dose_numeric_corrected.ge(0).all()

    result["dose_unit_corrected"] = result.dose_unit_reference
    result["metadata_source"] = SOURCE.name

    old_dose = pd.to_numeric(result.DOSE, errors="coerce")
    dose_changed = ~np.isclose(
        old_dose.to_numpy(float),
        result.dose_numeric_corrected.to_numpy(float),
        equal_nan=False,
    )

    result["dose_value_changed"] = dose_changed

    result.drop(columns="_merge").to_csv(
        OUT / f"{name}_metadata_corrected.csv",
        index=False,
    )

    print(f"\n=== {name} ===")
    print("接続したサンプル:", len(result))
    print("用量の数値を修正した行:", int(dose_changed.sum()))

    display(
        result.loc[
            dose_changed,
            [
                "COMPOUND_NAME",
                "DOSE_LEVEL",
                "DOSE",
                "dose_numeric_corrected",
                "DOSE_UNIT",
                "dose_unit_corrected",
            ],
        ].drop_duplicates().head(30)
    )

print("\n保存先:", OUT)

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

D = Path("/home/seowon1211/data1")
SOURCE = D / "tggates_metadata/Open-tggates_AllAttribute.tsv"
OUT = D / "conditioned_expression/metadata"
OUT.mkdir(parents=True, exist_ok=True)

assert SOURCE.exists(), f"파일 없음: {SOURCE}"

# 확인한 원본 파일의 인코딩
reference = pd.read_csv(
    SOURCE,
    sep="\t",
    encoding="cp932",
    dtype=str,
    keep_default_na=False,
)

required = {
    "BARCODE", "COMPOUND_NAME", "DOSE",
    "DOSE_UNIT", "DOSE_LEVEL", "SACRI_PERIOD",
}
assert required.issubset(reference.columns)


def barcode_key(series):
    # 숫자 식별자만 허용하고 앞자리 0 제거
    s = series.astype(str).str.strip()
    return s.str.lstrip("0").where(
        s.str.fullmatch(r"\d+") & s.str.lstrip("0").ne("")
    )


reference["barcode_key"] = barcode_key(reference.BARCODE)

# No ChipData 등 실제 칩 식별자가 없는 행 제외
reference = reference[
    reference.barcode_key.notna()
].copy()

assert not reference.barcode_key.duplicated().any(), (
    "원본 메타데이터에 BARCODE 중복이 있습니다."
)

reference = reference[
    [
        "barcode_key",
        "COMPOUND_NAME",
        "DOSE",
        "DOSE_UNIT",
        "DOSE_LEVEL",
        "SACRI_PERIOD",
    ]
].rename(columns={
    "COMPOUND_NAME": "compound_reference",
    "DOSE": "dose_reference",
    "DOSE_UNIT": "dose_unit_reference",
    "DOSE_LEVEL": "dose_level_reference",
    "SACRI_PERIOD": "time_reference",
})

for name in ["vitro", "vivo"]:
    path = D / f"{name}_train_test.csv"

    # 大容量の発現値は読み込まず、メタデータだけ取得
    meta = pd.read_csv(
        path,
        usecols=[
            "BARCODE",
            "COMPOUND_NAME",
            "SACRIFICE_PERIOD",
            "DOSE",
            "DOSE_UNIT",
            "DOSE_LEVEL",
            "usage",
        ],
        dtype=str,
        keep_default_na=False,
    )

    meta.insert(0, "expression_row", np.arange(len(meta)))
    meta["barcode_key"] = barcode_key(meta.BARCODE)

    assert meta.barcode_key.notna().all(), (
        f"{name}: 不正なBARCODEがあります。"
    )

    result = meta.merge(
        reference,
        on="barcode_key",
        how="left",
        validate="many_to_one",
        indicator=True,
        sort=False,
    )

    unmatched = result[result["_merge"].ne("both")]
    unmatched.to_csv(
        OUT / f"{name}_unmatched.csv", index=False
    )

    assert unmatched.empty, (
        f"{name}: 未接続が{len(unmatched)}行あります。"
    )
    assert np.array_equal(
        result.expression_row.to_numpy(),
        np.arange(len(meta)),
    ), "元データの行順が変わりました。"

    # 時間と用量等級が同じか確認
    for original, corrected in [
        ("SACRIFICE_PERIOD", "time_reference"),
        ("DOSE_LEVEL", "dose_level_reference"),
    ]:
        mismatch = (
            result[original].str.strip()
            != result[corrected].str.strip()
        )

        if mismatch.any():
            result.loc[mismatch].to_csv(
                OUT / f"{name}_{original}_mismatch.csv",
                index=False,
            )
            raise ValueError(
                f"{name}: {original}が一致しない行があります。"
            )

    result["dose_numeric_corrected"] = pd.to_numeric(
        result.dose_reference,
        errors="raise",
    )

    assert np.isfinite(result.dose_numeric_corrected).all()
    assert result.dose_numeric_corrected.ge(0).all()

    result["dose_unit_corrected"] = result.dose_unit_reference
    result["metadata_source"] = SOURCE.name

    old_dose = pd.to_numeric(result.DOSE, errors="coerce")
    dose_changed = ~np.isclose(
        old_dose.to_numpy(float),
        result.dose_numeric_corrected.to_numpy(float),
        equal_nan=False,
    )

    result["dose_value_changed"] = dose_changed

    result.drop(columns="_merge").to_csv(
        OUT / f"{name}_metadata_corrected.csv",
        index=False,
    )

    print(f"\n=== {name} ===")
    print("接続したサンプル:", len(result))
    print("用量の数値を修正した行:", int(dose_changed.sum()))

    display(
        result.loc[
            dose_changed,
            [
                "COMPOUND_NAME",
                "DOSE_LEVEL",
                "DOSE",
                "dose_numeric_corrected",
                "DOSE_UNIT",
                "dose_unit_corrected",
            ],
        ].drop_duplicates().head(30)
    )

print("\n保存先:", OUT)

In [8]:
from pathlib import Path

D = Path("/home/seowon1211/data1")

candidates = sorted(
    p for p in D.rglob("*")
    if p.is_file()
    and p.suffix.lower() == ".tsv"
    and "allattribute" in p.name.lower()
)

for p in candidates:
    print(p)

if len(candidates) == 1:
    SOURCE = candidates[0]
    print("\n사용할 파일:", SOURCE)
elif not candidates:
    print(
        "파일을 찾지 못했습니다. Jupyter의 파일 업로드 버튼으로 "
        "Open-tggates_AllAttribute.tsv를 data1 폴더에 올려주세요."
    )
else:
    print("\n여러 파일이 있습니다. 사용할 경로를 확인해주세요.")ㅋ

In [9]:
from pathlib import Path

D = Path("/home/seowon1211/data1")

candidates = sorted(
    p for p in D.rglob("*")
    if p.is_file()
    and p.suffix.lower() == ".tsv"
    and "allattribute" in p.name.lower()
)

for p in candidates:
    print(p)

if len(candidates) == 1:
    SOURCE = candidates[0]
    print("\n사용할 파일:", SOURCE)
elif not candidates:
    print(
        "파일을 찾지 못했습니다. Jupyter의 파일 업로드 버튼으로 "
        "Open-tggates_AllAttribute.tsv를 data1 폴더에 올려주세요."
    )
else:
    print("\n여러 파일이 있습니다. 사용할 경로를 확인해주세요.")

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd

D = Path("/home/seowon1211/data1")
SOURCE = D / "Open-tggates_AllAttribute.tsv"
OUT = D / "conditioned_expression/metadata"
OUT.mkdir(parents=True, exist_ok=True)

assert SOURCE.is_file(), f"파일 없음: {SOURCE}"

reference = pd.read_csv(
    SOURCE,
    sep="\t",
    encoding="cp932",
    dtype=str,
    keep_default_na=False,
)

required = {
    "BARCODE", "COMPOUND_NAME", "DOSE",
    "DOSE_UNIT", "DOSE_LEVEL", "SACRI_PERIOD",
}
missing = required - set(reference.columns)
assert not missing, f"원본 메타데이터에 필요한 컬럼 없음: {missing}"


def barcode_key(series):
    """숫자로 된 식별자의 앞자리 0만 제거합니다."""
    text = series.astype(str).str.strip()
    key = text.str.lstrip("0")
    return key.where(
        text.str.fullmatch(r"\d+") & key.ne("")
    )


reference["barcode_key"] = barcode_key(reference["BARCODE"])

# 칩 식별자가 없는 No ChipData 등의 행은 제외합니다.
reference = reference[
    reference["barcode_key"].notna()
].copy()

assert not reference["barcode_key"].duplicated().any(), (
    "원본 메타데이터에 중복 BARCODE가 있습니다."
)

reference = reference[
    [
        "barcode_key",
        "COMPOUND_NAME",
        "DOSE",
        "DOSE_UNIT",
        "DOSE_LEVEL",
        "SACRI_PERIOD",
    ]
].rename(columns={
    "COMPOUND_NAME": "compound_reference",
    "DOSE": "dose_reference",
    "DOSE_UNIT": "dose_unit_reference",
    "DOSE_LEVEL": "dose_level_reference",
    "SACRI_PERIOD": "time_reference",
})

for name in ["vitro", "vivo"]:
    # 대용량 발현값 대신 메타데이터 컬럼만 읽습니다.
    meta = pd.read_csv(
        D / f"{name}_train_test.csv",
        usecols=[
            "BARCODE",
            "COMPOUND_NAME",
            "SACRIFICE_PERIOD",
            "DOSE",
            "DOSE_UNIT",
            "DOSE_LEVEL",
            "usage",
        ],
        dtype=str,
        keep_default_na=False,
    )

    meta.insert(0, "expression_row", np.arange(len(meta)))
    meta["barcode_key"] = barcode_key(meta["BARCODE"])

    assert meta["barcode_key"].notna().all(), (
        f"{name}: 올바르지 않은 BARCODE가 있습니다."
    )
    assert not meta["barcode_key"].duplicated().any(), (
        f"{name}: 발현 데이터에 중복 BARCODE가 있습니다."
    )

    result = meta.merge(
        reference,
        on="barcode_key",
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )

    unmatched = result[result["_merge"].ne("both")]
    unmatched.to_csv(
        OUT / f"{name}_unmatched.csv",
        index=False,
    )

    assert unmatched.empty, (
        f"{name}: 연결하지 못한 샘플 {len(unmatched)}개. "
        "unmatched 파일을 확인해주세요."
    )

    assert np.array_equal(
        result["expression_row"].to_numpy(),
        np.arange(len(meta)),
    ), f"{name}: 발현 데이터의 행 순서가 달라졌습니다."

    # 시간과 용량 등급을 대조합니다.
    for original, corrected in [
        ("SACRIFICE_PERIOD", "time_reference"),
        ("DOSE_LEVEL", "dose_level_reference"),
    ]:
        mismatch = (
            result[original].str.strip()
            != result[corrected].str.strip()
        )

        if mismatch.any():
            result.loc[mismatch].to_csv(
                OUT / f"{name}_{original}_mismatch.csv",
                index=False,
            )
            raise ValueError(
                f"{name}: {original}이 일치하지 않습니다."
            )

    # 이름은 인코딩 차이가 있을 수 있어 기록만 남깁니다.
    name_mismatch = (
        result["COMPOUND_NAME"].str.strip()
        != result["compound_reference"].str.strip()
    )
    result.loc[name_mismatch].to_csv(
        OUT / f"{name}_compound_name_differences.csv",
        index=False,
    )

    result["dose_numeric_corrected"] = pd.to_numeric(
        result["dose_reference"],
        errors="raise",
    )

    corrected_dose = result[
        "dose_numeric_corrected"
    ].to_numpy(dtype=float)

    assert np.isfinite(corrected_dose).all()
    assert (corrected_dose >= 0).all()

    result["dose_unit_corrected"] = (
        result["dose_unit_reference"].str.strip()
    )
    assert result["dose_unit_corrected"].ne("").all()

    result["metadata_source"] = SOURCE.name

    old_dose = pd.to_numeric(
        result["DOSE"], errors="coerce"
    ).to_numpy(dtype=float)

    result["dose_value_changed"] = ~np.isclose(
        old_dose,
        corrected_dose,
        rtol=1e-9,
        atol=1e-12,
        equal_nan=False,
    )

    treated_zero = (
        ~result["DOSE_LEVEL"].str.lower().eq("control")
        & result["dose_numeric_corrected"].eq(0)
    )

    result.loc[treated_zero].to_csv(
        OUT / f"{name}_treated_zero_check.csv",
        index=False,
    )

    destination = OUT / f"{name}_metadata_corrected.csv"

    result.drop(columns="_merge").to_csv(
        destination,
        index=False,
    )

    print(f"\n=== {name} ===")
    print("연결한 샘플:", len(result))
    print(
        "용량 수치를 수정한 샘플:",
        int(result["dose_value_changed"].sum()),
    )
    print("화합물 이름 표기 차이:", int(name_mismatch.sum()))
    print("보정 후 용량이 0인 처리군:", int(treated_zero.sum()))
    print("저장:", destination)

    display(
        result.loc[
            result["dose_value_changed"],
            [
                "COMPOUND_NAME",
                "DOSE_LEVEL",
                "DOSE",
                "dose_numeric_corrected",
                "DOSE_UNIT",
                "dose_unit_corrected",
            ],
        ].drop_duplicates().head(30)
    )